# Notebook 00 — Configuración del Entorno SEPAL, Credenciales y AOIs Locales

- **Instancia SEPAL recomendada:** `t2` (para setup inicial) → luego usa `m8d` o `m16d` para procesar el stack y Random Forest a máxima velocidad en los siguientes notebooks.
- **Filosofía 100% SEPAL:** En este notebook solo descargamos **una única vez** tus dos polígonos `AOI_PACIFICO` y `AOI_CARIBE` (KB) y un recorte ligero `uint8` de `DinMangle_2020_2022_V23` (~MB) desde GEE hacia el disco local de SEPAL. A partir del Notebook 01 todo el cómputo pesado (Sentinel-2, Sentinel-1, Stack, PCA, Random Forest y mapas completos) corre **100% en las máquinas virtuales de SEPAL sin esperar descargas de GEE/Drive**.

In [ ]:
# 1. Instalar/actualizar dependencias (ejecutar una vez en la instancia SEPAL)
import os
import platform
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Python: {sys.version.split()[0]} | SO: {platform.system()} {platform.release()}")
print(f"CPUs disponibles: {os.cpu_count()} | Raiz del proyecto: {ROOT}")

for mount in [Path.home(), Path("/tmp")]:
    if mount.exists():
        u = shutil.disk_usage(mount)
        print(f"Disco {str(mount):18s} -> Libre: {u.free/1e9:7.1f} GB / Total: {u.total/1e9:7.1f} GB")

# Si falta alguna libreria nueva (ej. asf_search, pystac_client, rasterio), descomenta:
# %pip install -q -r ../requirements.txt

In [ ]:
# 2. Verificar librerias y detectar si ESA SNAP `gpt` esta disponible en SEPAL
import importlib
from src.local_s1 import detect_snap_gpt

PAQUETES = [
    "numpy", "pandas", "scipy", "yaml", "sklearn", "statsmodels",
    "matplotlib", "seaborn", "openpyxl", "PIL", "rasterio",
    "geopandas", "shapely", "pyproj", "pystac_client", "asf_search",
    "ee", "geemap", "ipywidgets", "ipyleaflet",
]
faltantes = []
for pkg in PAQUETES:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "ok")
        print(f"  [OK] {pkg:15s} ({ver})")
    except ImportError:
        faltantes.append(pkg)
        print(f"  [FALTA] {pkg}")

snap_gpt = detect_snap_gpt()
print(f"\nESA SNAP `gpt` detectado: {snap_gpt or 'No detectado en PATH (se usara OPERA_RTC_S1 en Python puro, recomendado)'}")
if faltantes:
    print("Ejecuta: %pip install -r ../requirements.txt y reinicia el kernel.")

In [ ]:
# 3. Cargar config.yaml y descargar UNA SOLA VEZ AOI_PACIFICO, AOI_CARIBE y DinMangle local
import geopandas as gpd
from src.config_utils import ensure_output_dirs, load_config
from src.local_aoi import (
    compute_region_grid,
    ensure_local_aoi,
    ensure_local_mangle_guide,
    get_local_aoi_path,
)

cfg = load_config()
dirs = ensure_output_dirs(cfg)

DESCARGAR_GUIA_DINMANGLE = True  # Descarga rapida (~5s por costa en computePixels) para uso 100% local en NB 02

resumen_local = []
for reg in ["PACIFICO", "CARIBE"]:
    aoi_gdf = ensure_local_aoi(reg, cfg)
    reg_cfg = cfg["regiones"][reg]
    grid = compute_region_grid(
        aoi_gdf,
        crs=reg_cfg["crs"],
        scale_m=float(cfg["stack"]["escala_m"]),
        buffer_m=float(cfg["stack"]["buffer_stack_m"]),
        margen_kernel_m=float(cfg["stack"]["margen_kernel_m"]),
    )
    guia_path = ensure_local_mangle_guide(reg, cfg, grid) if DESCARGAR_GUIA_DINMANGLE else None
    area_km2 = float(aoi_gdf.to_crs(reg_cfg["crs"]).area.sum() / 1e6)
    resumen_local.append({
        "region": reg,
        "crs": reg_cfg["crs"],
        "area_aoi_km2": round(area_km2, 2),
        "grilla_px": f"{grid['height']} x {grid['width']}",
        "aoi_gpkg": str(get_local_aoi_path(reg, cfg).name),
        "guia_tif": guia_path.name if guia_path else "No descargada",
    })

import pandas as pd
display(pd.DataFrame(resumen_local))

In [ ]:
# 4. Configurar credenciales gratuitas de NASA Earthdata Login (para Sentinel-1 en ASF)
# Si aun no tienes cuenta, creala gratis en 1 minuto en: https://urs.earthdata.nasa.gov/
# Luego en Earthdata Profile -> Applications -> Authorize 'Alaska Satellite Facility Data Access'.
from getpass import getpass
from src.local_s1 import has_earthdata_credentials, save_earthdata_netrc

ACTUALIZAR_CREDENCIALES_EARTHDATA = False

if has_earthdata_credentials() and not ACTUALIZAR_CREDENCIALES_EARTHDATA:
    print("[OK] Credenciales NASA Earthdata detectadas (~/.netrc o variables de entorno).")
else:
    if ACTUALIZAR_CREDENCIALES_EARTHDATA:
        usr = input("Usuario NASA Earthdata Login: ").strip()
        pwd = getpass("Contraseña NASA Earthdata Login (no se mostrara): ")
        netrc_p = save_earthdata_netrc(usr, pwd)
        print(f"[OK] Guardado de forma segura (chmod 600) en: {netrc_p}")
    else:
        print("[AVISO] Aun no tienes ~/.netrc de NASA Earthdata. Cambia ACTUALIZAR_CREDENCIALES_EARTHDATA = True para configurarlo antes de correr el Notebook 01.")

In [ ]:
# 5. Mapa de verificacion rapida con capas 100% locales en SEPAL
import geemap
from src.local_aoi import get_local_guia_path

m = geemap.Map(height="500px")
m.add_basemap("SATELLITE")

for reg, color in [("PACIFICO", "#FF8800"), ("CARIBE", "#00CCFF")]:
    gdf = ensure_local_aoi(reg, cfg)
    m.add_gdf(gdf, layer_name=f"AOI {reg}", style={"color": color, "fillOpacity": 0.1, "weight": 2})

m.setCenter(-90.35, 14.8, 7)
m